In [1]:
import pandas as pd
import sys
from pathlib import Path

In [6]:
PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

import src.data_utils as du

In [8]:
# Load dev set
df = du.load_pcl_tsv_to_df(du.pcl_path)
df = df.dropna(subset=['text', 'label'])
df = du.add_binary_label(df, threshold=2)

train_ids, dev_ids = du.load_split_ids()
_, official_dev_df = du.make_splits(df, train_ids, dev_ids)

print(f"Loaded {len(official_dev_df)} rows from dev")

Loaded 2094 rows from dev


In [10]:
pred_path = PROJECT_ROOT / "dev.txt"
print(f"Loading predictions from {pred_path.name}")

with open(pred_path, 'r') as f:
    preds = [int(line.strip()) for line in f.readlines() if line.strip()]

print(f"Loaded {len(preds)} predictions")

Loading predictions from dev.txt
Loaded 2094 predictions


In [13]:
analysis_df = official_dev_df.copy()
analysis_df['prediction'] = preds

false_positives = analysis_df[(analysis_df['binary_label'] == 0) & (analysis_df['prediction'] == 1)]

false_negatives = analysis_df[(analysis_df['binary_label'] == 1) & (analysis_df['prediction'] == 0)]

print("Error analysis")
print(f"Total dev samples : {len(analysis_df)}")
print(f"FP                : {len(false_positives)}")
print(f"FN                : {len(false_negatives)}")

Error analysis
Total dev samples : 2094
FP                : 112
FN                : 70


In [15]:
output_dir = PROJECT_ROOT / "Eval_and_Error"

fp_path = output_dir / "false_positives.csv"
fn_path = output_dir / "false_negatives.csv"

# Reorder columns
cols_to_save = ['par_id', 'keyword', 'country_code', 'text', 'binary_label', 'prediction']

false_positives[cols_to_save].to_csv(fp_path, index=False)
false_negatives[cols_to_save].to_csv(fn_path, index=False)


In [27]:
print("="*50)
print("Keyword distribution in errors (%)")
print("="*50)

fp_keywords = false_positives['keyword'].value_counts(normalize=True) * 100
fn_keywords = false_negatives['keyword'].value_counts(normalize=True) * 100

top_fp = fp_keywords.head(5)
top_fn = fn_keywords.head(5)

print("Top 5 keywords in false positives (model too sensitive):")
print((top_fp.round(1).astype(str) + '%').to_string())

print("\nTop 5 keywords in false negatives (model missed them):")
print((top_fn.round(1).astype(str) + '%').to_string())

print("\n" + "-"*50)
print("Overlap: keywords in both top 5 Lists")
print("-" * 50)

common_top_keywords = list(set(top_fp.index).intersection(set(top_fn.index)))

if common_top_keywords:
    overlap_df = pd.DataFrame({
        'False Positives (%)': top_fp.loc[common_top_keywords].round(1).astype(str) + '%',
        'False Negatives (%)': top_fn.loc[common_top_keywords].round(1).astype(str) + '%'
    })

    overlap_df = overlap_df.sort_values(by='False Positives (%)', ascending=False)
    print(overlap_df.to_string())
else:
    print("No overlapping keywords in the Top 5.")

Keyword distribution in errors (%)
Top 5 keywords in false positives (model too sensitive):
keyword
poor-families    19.6%
homeless         19.6%
hopeless         17.9%
in-need          17.0%
vulnerable        8.9%

Top 5 keywords in false negatives (model missed them):
keyword
poor-families    22.9%
women            14.3%
hopeless         14.3%
disabled         10.0%
homeless         10.0%

--------------------------------------------------
Overlap: keywords in both top 5 Lists
--------------------------------------------------
              False Positives (%) False Negatives (%)
keyword                                              
homeless                    19.6%               10.0%
poor-families               19.6%               22.9%
hopeless                    17.9%               14.3%


In [41]:
import textwrap
target_kw = 'poor-families'

print("="*80)
print(f"An example of incorrect predictions for: [{target_kw.upper()}]")
print("="*80)

fp_subset = false_positives[false_positives['keyword'] == target_kw]

if not fp_subset.empty:
    fp_sample = fp_subset.sample(1, random_state=42)
    fp_text = str(fp_sample['text'].iloc[0])

    print("False Positive: ")
    print(textwrap.fill(fp_text, width=80))
else:
    print(f"\nNo false positives found'{target_kw}'.")

fn_subset = false_negatives[false_negatives['keyword'] == target_kw]

if not fn_subset.empty:
    fn_sample = fn_subset.sample(1, random_state=42)
    fn_text = str(fn_sample['text'].iloc[0])

    print("\nFalse Negative: ")
    print(textwrap.fill(fn_text, width=80))
else:
    print(f"\nNo false negatives found for '{target_kw}'.")


An example of incorrect predictions for: [POOR-FAMILIES]
False Positive: 
Marcos said the government should help poor families that try every possible
means to survive . With Joel Zurbano <h> More from this Category :

False Negative: 
We are alarmed to learn of your recently circulated proposals that would
eviscerate the Lifeline program and leave many of the most vulnerable people in
the country without access to affordable communications . As you are well aware
, the Lifeline program provides a modest monthly subsidy of $9.25 to connect
low-income Americans to phone and internet services . As broadband prices
continue to soar , and affordability continues to suffer , adoption gaps remain
. The Lifeline program has proven critical for poor families and people of color
who are caught on the wrong side of the digital divide .
